        # 🔬 L03　抽樣與檢定
        **統計冒險之旅 2026**　｜　Day 1（09/05 六）🌄 統計之丘　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch5.2 bootstrap、Ch13 前言；資料：勇者咖啡八月銷售


        ### 🎯 這一關你會學到
        - 抽樣分布、中央極限定理、標準誤
- 信賴區間：公式版與 bootstrap 版
- p 值的直覺（置換檢定）、t 檢定、卡方檢定

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L03"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["3-1", "3-2", "3-3", "3-4", "3-5", "3-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_3_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "樣本平均"), 157.8333, 0.05): return (False, "樣本平均 不對：sample(30, random_state=42) 再 .mean()。")
    if not 約等於(抓變數(ns, "母體平均"), 140.8037, 0.05): return (False, "母體平均 = 金額.mean()。")
    return (約等於(抓變數(ns, "差距"), 17.0296, 0.1), "差距 = abs(樣本平均 - 母體平均)。")
任務定義("3-1", _check_3_1, 提示="random_state 要是 42。")

def _check_3_2(run):
    out, ns = run()
    if len(抓變數(ns, "樣本平均們")) != 1000: return (False, "要抽 1000 次（range(1000)）。")
    if not 約等於(抓變數(ns, "平均的平均"), 140.8362, 0.05): return (False, "平均的平均 = np.mean(樣本平均們)。")
    if not 約等於(抓變數(ns, "標準誤"), 17.5848, 0.05): return (False, "標準誤 = np.std(樣本平均們, ddof=1)。")
    return ("樣本平均" in " ".join(f["title"] for f in run.figs), "直方圖標題要包含「樣本平均」。")
任務定義("3-2", _check_3_2, 提示="range(1000)；np.mean；np.std(..., ddof=1)。")

def _check_3_3(run):
    out, ns = run()
    if not (約等於(抓變數(ns, "下界"), 129.7375, 0.2) and 約等於(抓變數(ns, "上界"), 160.4625, 0.2)): return (False, "公式版：m ± 1.96 * s / np.sqrt(n)。")
    return (約等於(抓變數(ns, "拔靴下界"), 129.7475, 0.5) and 約等於(抓變數(ns, "拔靴上界"), 160.0500, 0.5), "bootstrap 版：np.percentile(拔靴平均們, 2.5) 與 97.5；rng 種子要是 42。")
任務定義("3-3", _check_3_3, 提示="np.percentile(串列, 2.5)。")

def _check_3_4(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "觀察差"), 21.4232, 0.05): return (False, "觀察差 = 週末.mean() - 平日.mean()。")
    if int(抓變數(ns, "極端次數")) != 0: return (False, "極端次數 = (np.abs(洗牌差們) >= abs(觀察差)).sum()。")
    if not 約等於(抓變數(ns, "p值"), 1 / 1001, 1e-8): return (False, "有限次模擬請用修正公式：p值 = (1 + 極端次數) / (1000 + 1)。")
    return (str(抓變數(ns, "結論")) == "顯著", "p < 0.05 → 顯著。")
任務定義("3-4", _check_3_4, 提示="先數極端次數，再用 (1 + 極端次數) / (B + 1)；有限次模擬的 p 值不會等於 0。")

def _check_3_5(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "t值"), 5.5568, 0.05): return (False, "t值 不對：stats.ttest_ind(週末, 平日, equal_var=False)。")
    if float(抓變數(ns, "p值")) > 0.05: return (False, "p值 應該非常小。")
    return (str(抓變數(ns, "結論")) == "有差異", "p < 0.05 → 有差異。")
任務定義("3-5", _check_3_5, 提示="ttest_ind 回傳 (t, p) 兩個值。")

def _check_3_6(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "交叉表"), 列=3, 欄=3)
    if not ok: return (False, msg)
    if not 約等於(抓變數(ns, "卡方值"), 61.5358, 0.5): return (False, "卡方值 不對，交叉表要是 分店 × 類別。")
    return (str(抓變數(ns, "結論")) == "有關係", "p < 0.05 → 有關係。")
任務定義("3-6", _check_3_6, 提示="pd.crosstab(df['分店'], df['類別'])。")

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0-rc.1/data/coffee_sales_aug.csv")
金額 = df["金額"]

## 🔬 3-1　母體 vs 樣本：喝湯前先攪一攪
八月全部 2563 筆訂單是**母體（population）**——這次我們剛好有全部的資料。但真實世界通常只能拿到一部分：問 30 位客人、抽 100 張發票，這叫**樣本（sample）**。
> 🍲 喝一鍋湯不用整鍋喝完，攪一攪、舀一匙就知道鹹不鹹。統計就是「用一匙推論整鍋」的學問——關鍵是**這一匙有多準**。

In [ ]:
樣本 = 金額.sample(30, random_state=42)          # 隨機抽 30 筆
print("樣本平均", round(樣本.mean(), 1), "| 母體平均", round(金額.mean(), 1))
print("再抽一次（不同種子）：", round(金額.sample(30, random_state=7).mean(), 1))

## 3-2　抽樣分布與中央極限定理：很多匙的平均會越來越穩
每抽一次 30 筆，樣本平均都不一樣——這種「晃動」就是**抽樣變異**。抽 1,000 次、把 1,000 個樣本平均畫成直方圖，會看到：
1. 它們以母體平均為中心；
2. 形狀是**鐘形**（就算原始金額是右偏的！）——這就是**中央極限定理（CLT）**；
3. 晃動的大小＝**標準誤（standard error）**≈ σ / √n：樣本越大，晃得越小。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
樣本平均們 = [金額.sample(30, random_state=i).mean() for i in range(1000)]
print("1000 個樣本平均的平均", round(np.mean(樣本平均們), 1), "| 它們的標準差（標準誤）", round(np.std(樣本平均們, ddof=1), 2))
print("理論標準誤 σ/√n =", round(金額.std() / np.sqrt(30), 2))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(金額, bins=30, ax=ax[0]); ax[0].set_title("原始金額：右偏")
sns.histplot(樣本平均們, bins=30, ax=ax[1]); ax[1].set_title("1000 個樣本平均（n=30）：鐘形！")
plt.tight_layout(); plt.show()

## 3-3　信賴區間：漁網的寬度
只給一個樣本平均不老實，還要說「大概在哪個範圍」。**95% 信賴區間**≈ 樣本平均 ± 1.96 × 標準誤。
> 🎣 漁網：網越寬越可能網到真正的母體平均，但也越沒用。95% 的意思是：這樣撒 100 次網，大約 95 次會網到。

沒有公式怎麼辦？**bootstrap（拔靴法）**：把手上的樣本當成小母體，**抽後放回**再抽 1,000 次，看這 1,000 個平均散多開，取中間 95%。不需要任何理論假設。

In [ ]:
樣本 = 金額.sample(100, random_state=42)
n, m, s = len(樣本), 樣本.mean(), 樣本.std()
print("公式版 95% CI：", round(m - 1.96 * s / np.sqrt(n), 1), "~", round(m + 1.96 * s / np.sqrt(n), 1))
rng = np.random.default_rng(42)
拔靴平均們 = [rng.choice(樣本.values, size=n, replace=True).mean() for _ in range(1000)]
print("bootstrap 95% CI：", np.percentile(拔靴平均們, 2.5).round(1), "~", np.percentile(拔靴平均們, 97.5).round(1))
print("母體平均", round(金額.mean(), 1), "有沒有被網到？")

## 3-4　假設檢定：這是真的，還是巧合？
老闆說「週末每筆訂單金額比平日高」。差距是真的，還是抽樣晃動的巧合？

1. 先假設**沒差**（**虛無假設 H0**：週末＝平日）。
2. 問：如果真的沒差，**光靠運氣**看到這麼大的差距，機會有多少？這個機會就是 **p 值**。
3. p 值很小（習慣用 < 0.05）→ 「光靠運氣很難看到」→ 拒絕 H0，說差距**顯著**。

> 🃏 **置換檢定**：把「週末／平日」的名牌收回來洗牌重發 1,000 次，每次都算一次差距——這就是「如果沒差，光靠運氣會看到什麼」。
> ⚠️ p 值**不是**「H0 為真的機率」，也不代表差距很大。它只回答「這有多像巧合」。有限次模擬即使沒有任何一次同樣極端，也不能說 p=0；本關用 `(1 + 極端次數) / (B + 1)` 修正。

In [ ]:
週末 = df[df["星期"].isin(["星期六", "星期日"])]["金額"]
平日 = df[~df["星期"].isin(["星期六", "星期日"])]["金額"]
觀察差 = 週末.mean() - 平日.mean()
print("週末平均", round(週末.mean(), 1), "平日平均", round(平日.mean(), 1), "差距", round(觀察差, 1))
rng = np.random.default_rng(42)
全部 = 金額.values
洗牌差們 = []
for _ in range(1000):
    洗 = rng.permutation(全部)                      # 名牌洗牌重發
    洗牌差們.append(洗[:len(週末)].mean() - 洗[len(週末):].mean())
極端次數 = (np.abs(洗牌差們) >= abs(觀察差)).sum()
p值 = (1 + 極端次數) / (len(洗牌差們) + 1)
print("洗牌 1000 次，極端次數 =", 極端次數, "；修正後 p 值 =", p值)
sns.histplot(洗牌差們, bins=40); plt.axvline(觀察差, color="red"); plt.title("如果沒差，光靠運氣會看到的差距"); plt.show()

## 3-5　t 檢定與卡方檢定：現成的公式
置換檢定是「直覺版」，實務上用 `scipy.stats` 的現成檢定：
| 問題 | 檢定 | 寫法 |
|---|---|---|
| 兩組**平均**有沒有差？ | t 檢定 | `stats.ttest_ind(a, b, equal_var=False)` |
| 兩個**類別變數**有沒有關係？（分店 × 類別） | 卡方檢定 | `stats.chi2_contingency(pd.crosstab(x, y))` |

In [ ]:
t, p = stats.ttest_ind(週末, 平日, equal_var=False)
print("t 檢定：t =", round(t, 2), "p =", f"{p:.2e}")
交叉表 = pd.crosstab(df["分店"], df["類別"])
print(交叉表)
chi2, p2, dof, expected = stats.chi2_contingency(交叉表)
print("卡方檢定：卡方 =", round(chi2, 1), "p =", f"{p2:.2e}", "→ 各分店的品項組合", "有差" if p2 < 0.05 else "沒差")

### 🎯 任務 3-1　抽一個樣本

用 `random_state=42` 從金額抽 30 筆存成 `樣本`，算出 `樣本平均`、`母體平均`（全部金額的平均）與 `差距`（兩者相減的絕對值）。

In [ ]:
# 🎯 任務 3-1　抽一個樣本（請保留這一行）
樣本 = 金額.sample(30, random_state=???)
樣本平均 = ???
母體平均 = ???
差距 = abs(樣本平均 - 母體平均)
print(round(樣本平均, 1), round(母體平均, 1), round(差距, 1))

In [ ]:
檢查("3-1")   # ◀ 執行這一格，看看任務 3-1 有沒有過關

### 🎯 任務 3-2　抽樣分布模擬

抽 1,000 個樣本（每個 30 筆，第 i 次用 `random_state=i`），把樣本平均存成串列 `樣本平均們`；算出 `平均的平均`（`np.mean`）與 `標準誤`（`np.std(..., ddof=1)`），並畫出 `樣本平均們` 的直方圖（標題含「樣本平均」）。

In [ ]:
# 🎯 任務 3-2　抽樣分布模擬（請保留這一行）
樣本平均們 = [金額.sample(30, random_state=i).mean() for i in range(???)]
平均的平均 = ???
標準誤 = ???
sns.histplot(樣本平均們, bins=30); plt.title(???); plt.show()
print(round(平均的平均, 2), round(標準誤, 2), "理論值", round(金額.std() / np.sqrt(30), 2))

In [ ]:
檢查("3-2")   # ◀ 執行這一格，看看任務 3-2 有沒有過關

### 🎯 任務 3-3　兩種 95% 信賴區間

用 `random_state=42` 抽 100 筆存成 `樣本`。算出公式版的 `下界`、`上界`（平均 ± 1.96 × s/√n），以及 bootstrap 版的 `拔靴下界`、`拔靴上界`（種子 42 的 `rng`、抽後放回 1,000 次、取 2.5 與 97.5 百分位）。

In [ ]:
# 🎯 任務 3-3　兩種 95% 信賴區間（請保留這一行）
樣本 = 金額.sample(100, random_state=42)
n, m, s = len(樣本), 樣本.mean(), 樣本.std()
下界 = ???
上界 = ???
rng = np.random.default_rng(42)
拔靴平均們 = [rng.choice(樣本.values, size=n, replace=True).mean() for _ in range(1000)]
拔靴下界 = ???
拔靴上界 = ???
print(round(下界, 1), round(上界, 1), "|", round(拔靴下界, 1), round(拔靴上界, 1))

In [ ]:
檢查("3-3")   # ◀ 執行這一格，看看任務 3-3 有沒有過關

### 🎯 任務 3-4　置換檢定算 p 值

算出 `觀察差`（週末平均 − 平日平均），用種子 42 的 `rng` 洗牌 1,000 次得到 `洗牌差們`，先把 |洗牌差| ≥ |觀察差| 的個數存成 `極端次數`，再用有限次模擬修正 `p值 = (1 + 極端次數) / (1000 + 1)`。最後把 `結論` 設成 `"顯著"`（p < 0.05）或 `"不顯著"`。這個修正避免把有限次模擬的結果誤寫成 p=0。

In [ ]:
# 🎯 任務 3-4　置換檢定算 p 值（請保留這一行）
週末 = df[df["星期"].isin(["星期六", "星期日"])]["金額"]
平日 = df[~df["星期"].isin(["星期六", "星期日"])]["金額"]
觀察差 = ???
rng = np.random.default_rng(42)
全部 = 金額.values
洗牌差們 = []
for _ in range(1000):
    洗 = rng.permutation(全部)
    洗牌差們.append(洗[:len(週末)].mean() - 洗[len(週末):].mean())
極端次數 = ???
p值 = ???
結論 = ???
print(round(觀察差, 2), 極端次數, p值, 結論)

In [ ]:
檢查("3-4")   # ◀ 執行這一格，看看任務 3-4 有沒有過關

### 🎯 任務 3-5　t 檢定

對週末與平日的金額做 Welch t 檢定（`equal_var=False`），存成 `t值`、`p值`，並把 `結論` 設成 `"有差異"` 或 `"沒差異"`（p < 0.05）。

In [ ]:
# 🎯 任務 3-5　t 檢定（請保留這一行）
t值, p值 = stats.ttest_ind(週末, 平日, equal_var=???)
結論 = ???
print(round(t值, 2), f"{p值:.2e}", 結論)

In [ ]:
檢查("3-5")   # ◀ 執行這一格，看看任務 3-5 有沒有過關

### 🎯 任務 3-6　卡方檢定

做 `分店 × 類別` 的交叉表存成 `交叉表`，用 `chi2_contingency` 算出 `卡方值` 與 `p值`，`結論` 設成 `"有關係"` 或 `"沒關係"`。

In [ ]:
# 🎯 任務 3-6　卡方檢定（請保留這一行）
交叉表 = pd.crosstab(df[???], df[???])
卡方值, p值, 自由度, 期望 = stats.chi2_contingency(交叉表)
結論 = ???
print(交叉表)
print(round(卡方值, 1), f"{p值:.2e}", 結論)

In [ ]:
檢查("3-6")   # ◀ 執行這一格，看看任務 3-6 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把 3-2 的樣本大小從 30 改成 100，標準誤變成多少？和 σ/√n 的理論值比比看。
2. 用置換檢定比較「早上 vs 晚上」的金額，p 值多少？

---
## 🔑 通關密語
　你已經能分辨「真的有差」和「只是巧合」了——這是所有機器學習評估的起點。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⚔️ B1 Boss 戰：勇者咖啡 A/B 測試** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.1.0-rc.1/notebooks/B1_boss_ab_test.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/